# TekaRx Random Forest Model

This notebook trains a Random Forest classifier on the TekaRx FAERS cohort for drug safety signal detection.

> TekaRx outputs are research decision-support signals, not diagnoses or clinical advice. FAERS reports do not establish causality.

## Overview

This notebook:
1. Loads the processed cohort and features from parquet files
2. Prepares train/validation/test splits using prospective time-based splits
3. Trains a Random Forest classifier with appropriate hyperparameters
4. Evaluates performance with AUC-ROC, AUC-PR, and calibration metrics
5. Saves the trained model and feature importances

### Data Sources
- `data/processed/tekarx_cohort.parquet` — main cohort with labels
- `data/processed/case_splits.parquet` — prospective train/val/test splits
- `data/processed/drug_dictionary.parquet` — drug vocabulary
- `data/processed/edges/report_drug.parquet` — drug edges for drug-level features
- `data/processed/drug_risk_lookup.parquet` — precomputed drug risk scores
- `data/processed/dose_normalization_lookup.parquet` — normalized dose features
- `data/processed/indication_lookup.parquet` — indication features


## 0. Environment Setup

In [ ]:
import sys
from pathlib import Path

# Add src to path for tekarx imports
REPO_ROOT = Path().resolve().parent
sys.path.insert(0, str(REPO_ROOT / "src"))

import json
import warnings
from datetime import UTC, datetime
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    roc_curve,
    brier_score_loss,
    classification_report,
    confusion_matrix,
)
from sklearn.calibration import calibration_curve
from sklearn.model_selection import train_test_split
import joblib

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)

print(f"Python: {sys.version}")
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")
print(f"Scikit-learn: {__import__('sklearn').__version__}")

In [ ]:
DATA_DIR = REPO_ROOT / "data" / "processed"
INTERIM_DIR = REPO_ROOT / "data" / "interim"

COHORT_PATH = DATA_DIR / "tekarx_cohort.parquet"
SPLITS_PATH = DATA_DIR / "case_splits.parquet"
DRUG_DICT_PATH = DATA_DIR / "drug_dictionary.parquet"
DRUG_EDGES_PATH = DATA_DIR / "edges" / "report_drug.parquet"
DRUG_RISK_PATH = DATA_DIR / "drug_risk_lookup.parquet"
DOSE_LOOKUP_PATH = DATA_DIR / "dose_normalization_lookup.parquet"
INDICATION_PATH = DATA_DIR / "indication_lookup.parquet"

MODEL_DIR = REPO_ROOT / "models" / "random_forest"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

for p in [COHORT_PATH, SPLITS_PATH, DRUG_DICT_PATH, DRUG_EDGES_PATH, DRUG_RISK_PATH, DOSE_LOOKUP_PATH, INDICATION_PATH]:
    print(f"{'✓' if p.exists() else '✗'} {p.relative_to(REPO_ROOT)} ({p.stat().st_size / 1e6:.1f} MB)")

## 1. Load Data

In [ ]:
# Load cohort
cohort = pd.read_parquet(COHORT_PATH)
print(f"Cohort shape: {cohort.shape}")
print(f"Columns: {list(cohort.columns)}")
print(f"\nLabel distribution:")
print(cohort['y'].value_counts())
print(f"Positive rate: {cohort['y'].mean():.4f}")

cohort.head()

In [ ]:
# Load prospective splits
splits = pd.read_parquet(SPLITS_PATH)
print(f"Splits shape: {splits.shape}")
print(f"Columns: {list(splits.columns)}")
print(f"\nSplit distribution:")
print(splits['split'].value_counts())

splits.head()

In [ ]:
# Load lookup tables
drug_dict = pd.read_parquet(DRUG_DICT_PATH)
drug_edges = pd.read_parquet(DRUG_EDGES_PATH)
drug_risk = pd.read_parquet(DRUG_RISK_PATH)
dose_lookup = pd.read_parquet(DOSE_LOOKUP_PATH)
indication_lookup = pd.read_parquet(INDICATION_PATH)

for name, df in [
    ("drug_dict", drug_dict),
    ("drug_edges", drug_edges),
    ("drug_risk", drug_risk),
    ("dose_lookup", dose_lookup),
    ("indication_lookup", indication_lookup),
]:
    print(f"{name}: {df.shape} - columns: {list(df.columns)}")

## 2. Feature Engineering

In [ ]:
# Merge splits into cohort
cohort = cohort.merge(splits[['primaryid', 'split']], on='primaryid', how='left')
print(f"Cohort with splits: {cohort.shape}")
print(f"Split distribution in cohort:")
print(cohort['split'].value_counts())

# Check for missing splits
missing_splits = cohort['split'].isna().sum()
print(f"Missing splits: {missing_splits}")
if missing_splits > 0:
    cohort = cohort.dropna(subset=['split'])
    print(f"After dropping: {cohort.shape}")

In [ ]:
# Create drug-level features per report
# Count unique drugs per report
drug_counts = drug_edges.groupby('primaryid')['drug_concept_id'].nunique().reset_index()
drug_counts.columns = ['primaryid', 'n_drugs']

# Merge drug risk scores - average risk per report
drug_edges_risk = drug_edges.merge(drug_risk, on='drug_concept_id', how='left')
report_drug_risk = drug_edges_risk.groupby('primaryid')['risk_score'].agg(['mean', 'max', 'std']).reset_index()
report_drug_risk.columns = ['primaryid', 'drug_risk_mean', 'drug_risk_max', 'drug_risk_std']
report_drug_risk['drug_risk_std'] = report_drug_risk['drug_risk_std'].fillna(0)

# Drug class diversity (using struct2atc if available)
print(f"Drug edges with risk: {drug_edges_risk.shape}")
print(f"Report drug risk agg: {report_drug_risk.shape}")

In [ ]:
# Dose normalization features per report
dose_edges = pd.read_parquet(DATA_DIR / "edges" / "report_drug_dose.parquet")
print(f"Dose edges shape: {dose_edges.shape}")
print(f"Dose edges columns: {list(dose_edges.columns)}")

# Merge with dose lookup
dose_edges_norm = dose_edges.merge(dose_lookup, on='drug_concept_id', how='left')
print(f"Dose edges with lookup: {dose_edges_norm.shape}")

# Aggregate dose features per report
report_dose = dose_edges_norm.groupby('primaryid').agg(
    n_drugs_with_dose=('normalized_dose', 'count'),
    mean_normalized_dose=('normalized_dose', 'mean'),
    max_normalized_dose=('normalized_dose', 'max'),
    sum_normalized_dose=('normalized_dose', 'sum'),
).reset_index()
print(f"Report dose features: {report_dose.shape}")

In [ ]:
# Indication features - count unique indications per report
indi_path = INTERIM_DIR / "faers" / "indi"
indi_files = list(indi_path.glob("*.parquet"))
print(f"Indication files: {len(indi_files)}")

indi_dfs = []
for f in indi_files:
    df = pd.read_parquet(f, columns=['primaryid', 'indi_drug_seq', 'indi_pt'])
    indi_dfs.append(df)
indi_all = pd.concat(indi_dfs, ignore_index=True)

report_indication = indi_all.groupby('primaryid')['indi_pt'].nunique().reset_index()
report_indication.columns = ['primaryid', 'n_indications']
print(f"Report indications: {report_indication.shape}")

In [ ]:
# Demographic features from FAERS demo files
demo_path = INTERIM_DIR / "faers" / "demo"
demo_files = list(demo_path.glob("*.parquet"))
demo_dfs = []
for f in demo_files:
    df = pd.read_parquet(f, columns=['primaryid', 'age', 'sex', 'wt', 'reporter_country'])
    demo_dfs.append(df)
demo_all = pd.concat(demo_dfs, ignore_index=True)

# Keep latest demo per primaryid (deduplicate)
demo_latest = demo_all.drop_duplicates(subset='primaryid', keep='last')
print(f"Demo records: {demo_latest.shape}")
print(f"Age stats:\n{demo_latest['age'].describe()}")
print(f"Sex distribution:\n{demo_latest['sex'].value_counts()}")

# Encode sex
demo_latest['sex_encoded'] = demo_latest['sex'].map({'M': 1, 'F': 0, 'UNK': -1}).fillna(-1).astype(int)

# Age bins
demo_latest['age_group'] = pd.cut(demo_latest['age'], 
                                   bins=[0, 18, 40, 65, 85, 200],
                                   labels=['pediatric', 'young_adult', 'adult', 'elderly', 'very_elderly'])
demo_latest['age_group'] = demo_latest['age_group'].cat.add_categories('unknown').fillna('unknown')

In [ ]:
# Combine all features
feature_dfs = [
    cohort[['primaryid', 'y', 'split']],
    drug_counts,
    report_drug_risk,
    report_dose,
    report_indication,
    demo_latest[['primaryid', 'age', 'sex_encoded', 'age_group', 'wt', 'reporter_country']],
]

features = feature_dfs[0]
for df in feature_dfs[1:]:
    features = features.merge(df, on='primaryid', how='left')

# Fill missing values
numeric_cols = features.select_dtypes(include=[np.number]).columns
features[numeric_cols] = features[numeric_cols].fillna(0)

# Encode categorical
features = pd.get_dummies(features, columns=['age_group', 'reporter_country'], dummy_na=True)

print(f"Final features shape: {features.shape}")
print(f"Feature columns: {len([c for c in features.columns if c not in ['primaryid', 'y', 'split']])}")

## 3. Train/Validation/Test Split (Prospective)

In [ ]:
# Split by prospective time splits
train_mask = features['split'] == 'train'
val_mask = features['split'] == 'validation'
test_mask = features['split'] == 'test'

X_train = features[train_mask].drop(['primaryid', 'y', 'split'], axis=1)
y_train = features[train_mask]['y']

X_val = features[val_mask].drop(['primaryid', 'y', 'split'], axis=1)
y_val = features[val_mask]['y']

X_test = features[test_mask].drop(['primaryid', 'y', 'split'], axis=1)
y_test = features[test_mask]['y']

# Ensure same columns
X_val = X_val.reindex(columns=X_train.columns, fill_value=0)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

print(f"Train: {X_train.shape}, pos rate: {y_train.mean():.4f}")
print(f"Val:   {X_val.shape}, pos rate: {y_val.mean():.4f}")
print(f"Test:  {X_test.shape}, pos rate: {y_test.mean():.4f}")
print(f"\nFeature columns: {list(X_train.columns)}")

## 4. Random Forest Training

In [ ]:
# Random Forest configuration
RF_CONFIG = {
    "n_estimators": 500,
    "max_depth": 20,
    "min_samples_split": 10,
    "min_samples_leaf": 5,
    "max_features": "sqrt",
    "class_weight": "balanced_subsample",
    "n_jobs": -1,
    "random_state": 42,
    "verbose": 1,
}

rf = RandomForestClassifier(**RF_CONFIG)
print("Random Forest config:")
for k, v in RF_CONFIG.items():
    print(f"  {k}: {v}")

In [ ]:
# Train on training set
import time
start = time.time()
rf.fit(X_train, y_train)
train_time = time.time() - start
print(f"Training completed in {train_time:.1f} seconds")

## 5. Model Evaluation

In [ ]:
# Predict probabilities
y_train_pred = rf.predict_proba(X_train)[:, 1]
y_val_pred = rf.predict_proba(X_val)[:, 1]
y_test_pred = rf.predict_proba(X_test)[:, 1]

# Predict classes (threshold 0.5)
y_train_class = (y_train_pred >= 0.5).astype(int)
y_val_class = (y_val_pred >= 0.5).astype(int)
y_test_class = (y_test_pred >= 0.5).astype(int)

In [ ]:
def compute_metrics(y_true, y_pred_proba, y_pred_class, split_name):
    auc_roc = roc_auc_score(y_true, y_pred_proba)
    auc_pr = average_precision_score(y_true, y_pred_proba)
    brier = brier_score_loss(y_true, y_pred_proba)
    
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred_class).ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    ppv = tp / (tp + fp) if (tp + fp) > 0 else 0
    npv = tn / (tn + fn) if (tn + fn) > 0 else 0
    
    return {
        'split': split_name,
        'auc_roc': auc_roc,
        'auc_pr': auc_pr,
        'brier_score': brier,
        'sensitivity': sensitivity,
        'specificity': specificity,
        'ppv': ppv,
        'npv': npv,
        'tn': tn, 'fp': fp, 'fn': fn, 'tp': tp
    }

metrics = []
metrics.append(compute_metrics(y_train, y_train_pred, y_train_class, 'train'))
metrics.append(compute_metrics(y_val, y_val_pred, y_val_class, 'validation'))
metrics.append(compute_metrics(y_test, y_test_pred, y_test_class, 'test'))

metrics_df = pd.DataFrame(metrics)
print(metrics_df.to_string(index=False))

In [ ]:
# Plot ROC curves
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for split_name, y_true, y_pred in [
    ('Train', y_train, y_train_pred),
    ('Validation', y_val, y_val_pred),
    ('Test', y_test, y_test_pred),
]:
    fpr, tpr, _ = roc_curve(y_true, y_pred)
    auc = roc_auc_score(y_true, y_pred)
    axes[0].plot(fpr, tpr, label=f'{split_name} (AUC={auc:.3f})')
    
    precision, recall, _ = precision_recall_curve(y_true, y_pred)
    ap = average_precision_score(y_true, y_pred)
    axes[1].plot(recall, precision, label=f'{split_name} (AP={ap:.3f})')

axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.5)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curves')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curves')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(MODEL_DIR / 'roc_pr_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Calibration plot
fig, ax = plt.subplots(figsize=(8, 6))

for split_name, y_true, y_pred in [
    ('Train', y_train, y_train_pred),
    ('Validation', y_val, y_val_pred),
    ('Test', y_test, y_test_pred),
]:
    fraction_pos, mean_pred = calibration_curve(y_true, y_pred, n_bins=10, strategy='quantile')
    ax.plot(mean_pred, fraction_pos, 'o-', label=split_name)

ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Perfect calibration')
ax.set_xlabel('Mean Predicted Probability')
ax.set_ylabel('Fraction of Positives')
ax.set_title('Calibration Curves')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(MODEL_DIR / 'calibration.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Feature importance
importances = pd.DataFrame({
    'feature': X_train.columns,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

print("Top 20 features:")
print(importances.head(20).to_string(index=False))

# Plot top 20
plt.figure(figsize=(10, 8))
top20 = importances.head(20).iloc[::-1]
plt.barh(range(len(top20)), top20['importance'])
plt.yticks(range(len(top20)), top20['feature'])
plt.xlabel('Feature Importance')
plt.title('Top 20 Feature Importances (Random Forest)')
plt.tight_layout()
plt.savefig(MODEL_DIR / 'feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

# Save feature importances
importances.to_csv(MODEL_DIR / 'feature_importances.csv', index=False)

In [ ]:
# Confusion matrices
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, (split_name, y_true, y_pred) in enumerate([
    ('Train', y_train, y_train_class),
    ('Validation', y_val, y_val_class),
    ('Test', y_test, y_test_class),
]):
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx])
    axes[idx].set_title(f'{split_name} Confusion Matrix')
    axes[idx].set_xlabel('Predicted')
    axes[idx].set_ylabel('Actual')

plt.tight_layout()
plt.savefig(MODEL_DIR / 'confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Threshold Analysis

In [ ]:
# Threshold sweep for test set
thresholds = np.arange(0.1, 1.0, 0.02)
threshold_results = []

for thresh in thresholds:
    y_pred_thresh = (y_test_pred >= thresh).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred_thresh).ravel()
    
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    ppv = tp / (tp + fp) if (tp + fp) > 0 else 0
    f1 = 2 * ppv * sensitivity / (ppv + sensitivity) if (ppv + sensitivity) > 0 else 0
    
    threshold_results.append({
        'threshold': thresh,
        'sensitivity': sensitivity,
        'specificity': specificity,
        'ppv': ppv,
        'f1': f1,
        'tp': tp, 'fp': fp, 'tn': tn, 'fn': fn
    })

thresh_df = pd.DataFrame(threshold_results)

# Find optimal threshold (max F1)
best_idx = thresh_df['f1'].idxmax()
best_thresh = thresh_df.loc[best_idx, 'threshold']
best_f1 = thresh_df.loc[best_idx, 'f1']
print(f"Best threshold (max F1): {best_thresh:.3f}, F1: {best_f1:.4f}")
print(thresh_df.loc[best_idx])

# Plot threshold sweep
plt.figure(figsize=(10, 6))
plt.plot(thresh_df['threshold'], thresh_df['sensitivity'], label='Sensitivity')
plt.plot(thresh_df['threshold'], thresh_df['specificity'], label='Specificity')
plt.plot(thresh_df['threshold'], thresh_df['ppv'], label='PPV')
plt.plot(thresh_df['threshold'], thresh_df['f1'], label='F1', linewidth=2)
plt.axvline(best_thresh, color='red', linestyle='--', label=f'Best F1 ({best_thresh:.2f})')
plt.xlabel('Threshold')
plt.ylabel('Score')
plt.title('Threshold Sweep (Test Set)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(MODEL_DIR / 'threshold_sweep.png', dpi=150, bbox_inches='tight')
plt.show()

thresh_df.to_csv(MODEL_DIR / 'threshold_sweep.csv', index=False)

## 7. Save Model & Artifacts

In [ ]:
# Save model
model_path = MODEL_DIR / "random_forest_model.joblib"
joblib.dump(rf, model_path)
print(f"Model saved to: {model_path}")

# Save metrics
metrics_df.to_csv(MODEL_DIR / 'metrics.csv', index=False)

# Save config
import json
with open(MODEL_DIR / 'config.json', 'w') as f:
    json.dump({
        'model_type': 'RandomForestClassifier',
        'config': RF_CONFIG,
        'feature_columns': list(X_train.columns),
        'n_features': len(X_train.columns),
        'train_samples': len(X_train),
        'val_samples': len(X_val),
        'test_samples': len(X_test),
        'train_pos_rate': float(y_train.mean()),
        'val_pos_rate': float(y_val.mean()),
        'test_pos_rate': float(y_test.mean()),
        'best_threshold': float(best_thresh),
        'timestamp': datetime.now(UTC).isoformat(),
    }, f, indent=2)

# Save test predictions for further analysis
test_predictions = pd.DataFrame({
    'primaryid': features[test_mask]['primaryid'].values,
    'y_true': y_test.values,
    'y_pred_proba': y_test_pred,
    'y_pred_class': y_test_class,
    'y_pred_class_optimal': (y_test_pred >= best_thresh).astype(int)
})
test_predictions.to_parquet(MODEL_DIR / 'test_predictions.parquet', index=False)

print("\nAll artifacts saved to:", MODEL_DIR)

## 8. Summary

In [ ]:
print("=" * 60)
print("RANDOM FOREST MODEL SUMMARY")
print("=" * 60)
print(f"Model: RandomForestClassifier (n_estimators={RF_CONFIG['n_estimators']}, max_depth={RF_CONFIG['max_depth']})")
print(f"Features: {len(X_train.columns)}")
print(f"Train samples: {len(X_train):,} (pos rate: {y_train.mean():.4f})")
print(f"Val samples:   {len(X_val):,} (pos rate: {y_val.mean():.4f})")
print(f"Test samples:  {len(X_test):,} (pos rate: {y_test.mean():.4f})")
print(f"\nTest Metrics:")
test_metrics = metrics_df[metrics_df['split'] == 'test'].iloc[0]
print(f"  AUC-ROC:  {test_metrics['auc_roc']:.4f}")
print(f"  AUC-PR:   {test_metrics['auc_pr']:.4f}")
print(f"  Brier:    {test_metrics['brier_score']:.4f}")
print(f"  Sensitivity: {test_metrics['sensitivity']:.4f}")
print(f"  Specificity: {test_metrics['specificity']:.4f}")
print(f"  PPV:         {test_metrics['ppv']:.4f}")
print(f"  NPV:         {test_metrics['npv']:.4f}")
print(f"\nBest threshold (F1): {best_thresh:.3f}")
print(f"Model saved to: {MODEL_DIR}")
print("=" * 60)